# Step 1: Load features and LightGBM

- Running features.ipynb file to get students and teacher tables

In [2]:

from google.colab import drive
drive.mount('/content/drive')

%run /content/drive/MyDrive/m5_project/notebooks/features.ipynb

import lightgbm as lgb
print('lightgbm', lgb.__version__)
print('teacher table:', teacher.shape, '| levels:', teacher.level.unique().tolist())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(6096944, 8)
days: 847 -> 1941
L1_total            1 series
L2_store            2 series
L3_cat              3 series
L4_dept             7 series
L5_store_cat        6 series
L6_store_dept      14 series
L7_item          3049 series
L8_item_store    6098 series
(6098, 1095)
L1_total            1 series x 1095 days
L2_store            2 series x 1095 days
L3_cat              3 series x 1095 days
L4_dept             7 series x 1095 days
L5_store_cat        6 series x 1095 days
L6_store_dept      14 series x 1095 days
L7_item          3049 series x 1095 days
L8_item_store    6098 series x 1095 days
Level 1: Sum of both the stores
d       847   848   849   850   851
Total  7745  8747  7996  8197  6266
--------

# Step 2: Train / validation / test split (teacher)

In [11]:
FEATURES = (['series_id']
            + [f'lag_{l}' for l in LAGS]
            + [f'roll_mean_{w}' for w in WINDOWS]
            + ['wday', 'month', 'day_of_month', 'event_type', 'snap_CA']
            )
teacher['series_id'] = teacher['series_id'].astype('category')

t_train = teacher[teacher.d <= TRAIN_END]
t_valid = teacher[(teacher.d >= VAL_START) & (teacher.d < TEST_START)]
t_test  = teacher[teacher.d >= TEST_START]

print('train:', t_train.shape, '| valid:', t_valid.shape, '| test:', t_test.shape)
assert len(t_train) + len(t_valid) + len(t_test) == len(teacher)

train: (12428, 18) | valid: (364, 18) | test: (364, 18)


# Step 3 :Train one teacher and check it on validation

- One LightGBM (tweedie loss) trained on all 13 teacher series

In [14]:
def level_score(rows, preds, last_day):
  scales, weights = get_scales(last_day), get_weights(last_day)
  rows = rows.assign(pred=preds)
  out = {}
  for name in TEACHER_LEVELS:
      part   = rows[rows.level == name]
      pred   = part.pivot(index='series_id', columns='d', values='pred')   # series x days
      actual = sales_by_level[name][pred.columns]
      pred   = pred.reindex(actual.index)                                  # same row order
      rmsse  = np.sqrt(((actual - pred) ** 2).mean(axis=1) / scales[name])
      out[name] = (weights[name] * rmsse).sum()
  return pd.Series(out)

params = dict(objective='tweedie', learning_rate=0.05, num_leaves=31,
              min_child_samples=20, n_estimators=500, random_state=42, verbose=-1)

model = lgb.LGBMRegressor(**params)
model.fit(t_train[FEATURES], t_train['sales'])

val_scores = level_score(t_valid, model.predict(t_valid[FEATURES]), TRAIN_END)
print(val_scores.round(3))
print('teacher validation score (mean of L1-L4):', round(val_scores.mean(), 3))

L1_total    0.312
L2_store    0.420
L3_cat      0.297
L4_dept     0.394
dtype: float64
teacher validation score (mean of L1-L4): 0.356


In [15]:
# Explanation of above output
# even with normal, untuned settings, the teacher beats the lazy “copy last month” guess by almost half at every level. It has really learned the weekly pattern, the trend and the calendar effects.

# Step 4: TCV (time-series Cross Validation), trying many teacher settings and keeping the best

- Getting best LightGBM settings for the teacher, using normal TCV

In [16]:
# Different tuning Knobs for model

SPACE = {
    'learning_rate':          [0.01, 0.03, 0.05, 0.1],
    'num_leaves':             [15, 31, 63, 127],
    'min_child_samples':      [5, 10, 20, 50],
    'n_estimators':           [200, 500, 1000],
    'colsample_bytree':       [0.6, 0.8, 1.0],
    'tweedie_variance_power': [1.1, 1.3, 1.5],
}
N_TRIALS = 30
rng = np.random.default_rng(42)

trials = []
for t in range(N_TRIALS):
    p = {name: rng.choice(options).item() for name, options in SPACE.items()}   # pick a random strategy at every run
    model = lgb.LGBMRegressor(objective='tweedie', random_state=42, verbose=-1, **p)
    model.fit(t_train[FEATURES], t_train['sales'])
    s = level_score(t_valid, model.predict(t_valid[FEATURES]), TRAIN_END)
    trials.append({'trial': t, 'val_score': s.mean(), **s.round(3).to_dict(), 'params': p})
    print(f'trial {t:>2}  val_score {s.mean():.3f}')

trials = pd.DataFrame(trials).sort_values('val_score')  # Best grades first
best_params = trials.iloc[0]['params']                  # keep the wining

print(trials.head(5)[['trial', 'val_score', 'L1_total', 'L2_store', 'L3_cat', 'L4_dept']].to_string(index=False))
print('best teacher settings:', best_params)


trial  0  val_score 0.390
trial  1  val_score 0.926
trial  2  val_score 0.400
trial  3  val_score 0.348
trial  4  val_score 0.385
trial  5  val_score 0.389
trial  6  val_score 0.385
trial  7  val_score 0.374
trial  8  val_score 0.349
trial  9  val_score 0.424
trial 10  val_score 0.397
trial 11  val_score 0.388
trial 12  val_score 0.364
trial 13  val_score 0.354
trial 14  val_score 0.385
trial 15  val_score 0.345
trial 16  val_score 0.812
trial 17  val_score 0.329
trial 18  val_score 0.372
trial 19  val_score 0.379
trial 20  val_score 0.368
trial 21  val_score 0.403
trial 22  val_score 0.797
trial 23  val_score 0.386
trial 24  val_score 0.435
trial 25  val_score 0.349
trial 26  val_score 0.373
trial 27  val_score 0.362
trial 28  val_score 0.425
trial 29  val_score 0.378
 trial  val_score  L1_total  L2_store  L3_cat  L4_dept
    17   0.328602     0.274     0.336   0.306    0.399
    15   0.344886     0.298     0.395   0.300    0.386
     3   0.347786     0.299     0.419   0.305    0.368


# Step 5 — Proxy forecasts (core of H-Pro)
- Best teacher (Cell 4 TCV) retrained on d_930–1913, then forecasts the test month d_1914–1941 for all 13 series of L1–L4

In [17]:
t_known = teacher[teacher.d < TEST_START]  # days, d_930–1913

# building teacher model with the "trail 17- winner"
final_teacher = lgb.LGBMRegressor(objective='tweedie', random_state=42, verbose=-1, **best_params)
final_teacher.fit(t_known[FEATURES], t_known['sales'])

t_test = t_test.assign(proxy=final_teacher.predict(t_test[FEATURES]))

proxies = {}
for name in TEACHER_LEVELS:
    part = t_test[t_test.level == name]
    proxies[name] = (part.pivot(index='series_id', columns='d', values='proxy').reindex(sales_by_level[name].index))

for name, table in proxies.items():
    print(f'{name:<10} {table.shape}   missing: {table.isna().sum().sum()}')

OUT = DRIVE / 'outputs'
OUT.mkdir(exist_ok=True)
saved = pd.concat(proxies, names=['level', 'series_id'])
saved.columns = saved.columns.astype(str)                 # parquet needs text column names
saved.to_parquet(OUT / 'teacher_proxies.parquet')
print('saved:', saved.shape)

proxies['L1_total'].round(0).iloc[:, :7]

L1_total   (1, 28)   missing: 0
L2_store   (2, 28)   missing: 0
L3_cat     (3, 28)   missing: 0
L4_dept    (7, 28)   missing: 0
saved: (13, 28)


d,1914,1915,1916,1917,1918,1919,1920
Total,7933.0,7244.0,7079.0,7151.0,8685.0,13543.0,11060.0


# Step 6 — Teacher test RMSSE per level

- Proxies compared with the real test sales d_1914–1941

In [18]:
test_scores = level_score(t_test, t_test['proxy'], TEST_START - 1)
print(test_scores.round(3))
print('teacher test score (mean of L1-L4):', round(test_scores.mean(), 3))

test_scores.rename('teacher_test_rmsse').to_csv(OUT / 'teacher_test_rmsse.csv')

L1_total    0.577
L2_store    0.681
L3_cat      0.612
L4_dept     0.667
dtype: float64
teacher test score (mean of L1-L4): 0.634


In [ ]:
# The teacher's proxies are about twice as wrong on the test month as on validation, showing the validation-vs-test gap the paper describes